1. Imports & Setup:

Loads pandas for data manipulation and suppresses warnings for cleaner output.

In [1]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


Libraries imported successfully!


2. Installed Generation Capacity & Solar Share (2024)

Queries the ENTSO-E API for installed generation capacity for selected countries
(AT, BE, BG, CZ, FR, GR, HU, ES), extracting total capacity and solar capacity in MW per country.
Solar share (%) is computed as solar capacity divided by total capacity, and results are sorted descending by solar share, while countries with empty responses or API errors are logged in failed.

In [3]:
from entsoe import EntsoePandasClient
import pandas as pd

client = EntsoePandasClient(api_key="7308a46a-2e78-4bf2-a552-3295649e49e3")

eu_countries = ["AT", "BE", "BG", "CZ", "FR", "GR", "HU", "ES"]

start = pd.Timestamp("2024-01-02", tz="UTC")
end = pd.Timestamp("2024-01-03", tz="UTC")

rows = []
failed = []

for i, cc in enumerate(eu_countries, start=1):
    print(f"[{i}/{len(eu_countries)}] {cc} ...", end=" ")
    try:
        df = client.query_installed_generation_capacity(cc, start=start, end=end)
    except Exception as e:
        print(f"error: {e}")
        failed.append(cc)
        continue

    if df.empty:
        print("no data")
        failed.append(cc)
        continue

    row = df.iloc[0]
    total_cap = float(row.sum())
    solar_cap = float(row.get("Solar", 0.0))

    rows.append(
        {
            "country": cc,
            "total_capacity_MW": total_cap,
            "solar_capacity_MW": solar_cap,
        }
    )
    print("ok")


country_caps = pd.DataFrame(rows)

if not country_caps.empty:
    country_caps["solar_share_pct"] = (
        100 * country_caps["solar_capacity_MW"] / country_caps["total_capacity_MW"]
    )
    print("\nResult:")
    print(
        country_caps[
            ["country", "total_capacity_MW", "solar_capacity_MW", "solar_share_pct"]
        ].sort_values("solar_share_pct", ascending=False)
    )
else:
    print("country_caps is empty – no successful data rows were collected.")
    print("Failed countries:", failed)


[1/8] AT ... ok
[2/8] BE ... ok
[3/8] BG ... ok
[4/8] CZ ... ok
[5/8] FR ... ok
[6/8] GR ... ok
[7/8] HU ... ok
[8/8] ES ... ok

Result:
  country  total_capacity_MW  solar_capacity_MW  solar_share_pct
6      HU        11222.90600           4029.077        35.900479
1      BE        28114.54000           8788.790        31.260657
2      BG        15848.00000           4568.000        28.823826
5      GR        24122.00000           6700.000        27.775475
0      AT        28994.25000           7294.000        25.156712
7      ES       116949.10000          23867.300        20.408280
3      CZ        21198.00000           3548.000        16.737428
4      FR       148464.86694          17794.870        11.985913
